# 👥 Notebook 2: User-Based Collaborative Filtering

**Mục tiêu:**
- Hiểu ý tưởng: "Người dùng giống nhau sẽ thích phim giống nhau"
- Tính similarity giữa users (Cosine)
- Tìm K nearest neighbors
- Gợi phim dựa trên neighbors

**⚠️ Chạy notebook 00 hoặc 01 trước để có data!**

## 1. Lý thuyết: User-Based CF

```
Ý tưởng: Nếu User A và User B đều thích phim X, Y, Z
         → Họ có "gu" giống nhau
         → Phim mà User B thích (mà A chưa xem)
           rất có thể User A cũng sẽ thích

Các bước:
  1. Tính similarity giữa User A và tất cả users khác
  2. Chọn K users giống A nhất (K-Nearest Neighbors)
  3. Dự đoán rating của A cho phim i:
     r̂(A,i) = r̄_A + Σ(sim(A,B) × (r_B,i - r̄_B)) / Σ|sim(A,B)|
```

In [ ]:
import subprocess, sys, os

try:
    import surprise
    import numpy as np
    import pandas as pd
    assert int(np.__version__.split('.')[0]) < 2, 'need numpy<2'
    assert int(pd.__version__.split('.')[0]) < 3, 'need pandas<3'
    print(f'✅ OK (numpy={np.__version__}, pandas={pd.__version__}, surprise={surprise.__version__})')
except Exception as e:
    print(f'📦 Installing... ({e})')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        'numpy<2', 'pandas<3', 'scikit-surprise', 'scikit-learn',
        'matplotlib', 'seaborn', 'tqdm', '-q'])
    print('✅ Install xong! Runtime đang restart...')
    os.kill(os.getpid(), 9)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from surprise import KNNWithMeans, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split, cross_validate
import warnings
warnings.filterwarnings('ignore')

os.makedirs('results/charts', exist_ok=True)

ratings = pd.read_csv('data/processed/ratings_clean.csv')
movies  = pd.read_csv('data/processed/movies_clean.csv')

print(f'Loaded: {len(ratings):,} ratings, {len(movies):,} movies')

## 2. Chuẩn bị data cho scikit-surprise

In [ ]:
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print(f'Train: {trainset.n_ratings:,} | Test: {len(testset):,}')

## 3. Train User-Based CF Model

In [ ]:
DEFAULT_K = 40

model = KNNWithMeans(
    k=DEFAULT_K,
    sim_option={'name': 'cosine', 'user_based': True},
    verbose=False
)
model.fit(trainset)
print('✅ Train xong!')

## 4. Đánh giá trên Test Set

In [ ]:
predictions = model.test(testset)
rmse = accuracy.rmse(predictions, verbose=False)
mae  = accuracy.mae(predictions, verbose=False)

print(f'📊 User-Based CF:')
print(f'   RMSE: {rmse:.4f}')
print(f'   MAE:  {mae:.4f}')

## 5. Cross-Validation 5-fold

In [ ]:
cv_results = cross_validate(model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)
print(f"\nCV RMSE: {cv_results['test_rmse'].mean():.4f} ± {cv_results['test_rmse'].std():.4f}")
print(f"CV MAE:  {cv_results['test_mae'].mean():.4f} ± {cv_results['test_mae'].std():.4f}")

## 6. Gợi ý phim cho User

In [ ]:
def recommend_user_cf(model, user_id, ratings_df, movies_df, top_n=10):
    """Gợi phim bằng User-Based CF."""
    user_movies = ratings_df[ratings_df['userId'] == user_id]['movieId'].values
    all_movies = ratings_df['movieId'].unique()
    unseen = [m for m in all_movies if m not in user_movies]
    scores = {m: model.predict(user_id, m).est for m in unseen}
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    results = []
    for mid, score in sorted_scores[:top_n]:
        title = movies_df[movies_df['movieId'] == mid]['title'].values
        if len(title) > 0:
            results.append({'movieId': mid, 'title': title[0], 'score': round(score, 2)})
    return results

recs = recommend_user_cf(model, 1, ratings, movies, top_n=10)
print('🎬 Top 10 gợi ý cho User 1:')
pd.DataFrame(recs)

## 7. Thử nghiệm K

In [ ]:
k_values = [10, 20, 40, 60, 80]
rmse_scores = []

for k in k_values:
    m = KNNWithMeans(k=k, sim_option={'name': 'cosine', 'user_based': True}, verbose=False)
    cv = cross_validate(m, data, measures=['RMSE'], cv=3, verbose=False)
    rmse_scores.append(cv['test_rmse'].mean())
    print(f'  K={k:3d} → RMSE={cv["test_rmse"].mean():.4f}')

plt.figure(figsize=(8, 4))
plt.plot(k_values, rmse_scores, marker='o', color='steelblue', linewidth=2, markersize=8)
plt.xlabel('K (số neighbors)')
plt.ylabel('RMSE')
plt.title('User-Based CF: RMSE theo K')
plt.grid(True, alpha=0.3)
plt.savefig('results/charts/02_user_cf_rmse_vs_k.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Tổng kết

- RMSE ≈ 0.87–0.90, MAE ≈ 0.68–0.70
- K quá nhỏ → thiếu thông tin, K quá lớn → nhiễu
- Thường dùng K=20–40 cho MovieLens